# Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from llm.interface.qwen import Qwen

In [2]:
model = Qwen("llm/weight/qwen25-7b")
model.load_model()
model.load_tokenizer()

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
import sqlite3
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [4]:
from tqdm import tqdm
import pandas as pd
DATA_SRC = '../../data_src'
tables = {
    'Table_0': 'pandas_dfs/codebase_community/comments.csv',
    'Table_1': 'pandas_dfs/codebase_community/posts.csv',
    'Table_2': 'pandas_dfs/california_schools/satscores.csv',
}
for table in tqdm(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}')
    df.to_sql(table, conn, index=False, if_exists="replace")

100%|██████████| 3/3 [00:08<00:00,  2.94s/it]


In [5]:
question = "Among the schools with the average score in Math over 560 in the SAT test, how many schools are in the bay area?"
target_schema = "['School ID', 'School Name', 'Average Math Score', 'Location']"

# Plan Generation

## Zero-th Step

In [ ]:
from llm.prompts import clear_schema_system_prompt
from utils import format_schema_with_samples

updated_schemas: list[str] = []
for table in tables:
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}')
    first_step_msg = [
        {'role': 'system', 'content': clear_schema_system_prompt},
        {'role': 'user', 'content': f'Schema: {format_schema_with_samples(df)}'}
    ]
    output = model.chat(first_step_msg)
    updated_schemas.append(output)

In [ ]:
for i in updated_schemas:
    print(i)

## First Step

In [6]:
updated_schemas = [
    "['Post ID', 'Score', 'Text', 'Creation Date', 'User ID', 'User Display Name']",
    "['QuestionId', 'PostTypeId', 'AcceptedAnswerId', 'CreationDate', 'Score', 'ViewCount', 'Body', 'OwnerUserId', 'LastActivityDate', 'Title', 'Tags', 'AnswerCount', 'CommentCount', 'FavoriteCount', 'LastEditorUserId', 'LastEditDate', 'CommunityOwnedDate', 'ParentId', 'ClosedDate', 'OwnerDisplayName', 'LastEditorDisplayName']",
    "['cds_number', 'school_type', 'school_name', 'district_name', 'county_name', 'total_enrollment_12th_grade', 'number_of_test_takers', 'average_reading_score', 'average_math_score', 'average_writing_score', 'number_of_grades_1500_or_above']",
]

In [7]:
from utils import format_schema
from ast import literal_eval
available_tables = ""
for table_idx, table in enumerate(tables):
    df = pd.read_csv(f'{DATA_SRC}/{tables[table]}', names=literal_eval(updated_schemas[table_idx]), skiprows=1)
    available_tables += f"\n{table}: ```{format_schema(df)}```\n"

In [8]:
from llm.prompts import plan_generator_first_step_system_prompt
first_step_msg = [
    {'role': 'system', 'content': plan_generator_first_step_system_prompt},
    {'role': 'user', 'content': f'Question: {question}\nAvailable Tables: {available_tables}\nTarget Schema: {target_schema}'}
]

In [10]:
first_step_out = model.chat(first_step_msg)

In [11]:
print(first_step_out)

{
    "operation": "select_table",
    "tables_involved": ["Table_2"],
    "description": "Select Table_2."
}


## Step 1.5: Get the Base Table

In [ ]:
first_step_out = {
    "operation": "select_table",
    "tables_involved": ["Table_2"],
    "description": "Select Table_2."
}

## Step 2: Column Projection

In [ ]:
plan = [
    {'operation': 'select_table', 'description': 'Select Table_2 as the base of subsequent operations.'}
]

In [ ]:
model.chat([{'role': 'user', 'content': f'Convert this description: {plan[0]['description']} to SQL code. Please answer directly; ensure that your output can be parsed directly as SQL code.'}])

In [ ]:
plan = [
    {'operation': 'join_table', 'description': 'Join Table_1 and Table_2'}
]

## Execution of Select Table

In [ ]:
query = "SELECT * FROM Table_2;"
result = pd.read_sql_query(query, conn)
result.head()

## Execution of Join Table

In [ ]:
query = "SELECT * FROM Table_2;"
result = pd.read_sql_query(query, conn)
result.head()